# Transactions

Some operations involve several steps that must all succeed together. Lending
a book means two changes: record the loan and mark the book as unavailable.
If the first succeeds but the second fails, your data is left inconsistent. A
**transaction** groups statements so that either all of them apply or none
of them do. This notebook teaches transactions and adds the final pieces to
your Library: staff and individual book copies.

## What a transaction does, before the SQL

The scenario is a common library operation: lending a book. It records a loan and marks the book unavailable, as a single unit of work.

Step through it and watch two things:

- Another librarian, working on a separate connection, sees none of your changes while your transaction is open. This is isolation.
- Choose Commit and your changes become permanent and visible to everyone. Choose Roll back and every change is undone, as if the transaction never happened. This is the "all or nothing" guarantee.

The SQL under each step is the syntax you will learn in the rest of this notebook. For now, focus on what happens, not how it is written.

## BEGIN, COMMIT, ROLLBACK

A transaction starts with BEGIN and ends with either COMMIT (apply all
changes) or ROLLBACK (discard all changes):

```sql
BEGIN;
... several statements ...
COMMIT;     -- or ROLLBACK to undo everything since BEGIN
```

If anything goes wrong before COMMIT, you ROLLBACK and the database is
exactly as it was before BEGIN. This is what keeps multi-step operations
safe.

In [ ]:
from rel_db_learning.widgets import transaction_playground

transaction_playground()

In [ ]:
from rel_db_learning import get_project_connection, run_sql, run_ddl, reset_my_project_data

project = get_project_connection()
reset_my_project_data()

## Growing the Library: staff and copies

Before we practice transactions on realistic data, let's add the final
schema pieces:

- A **librarian** table: the staff who process loans. Some are managers.
- A **book_copy** table: libraries own several physical copies of popular
  books, each tracked separately.
- A link from **loan** to **librarian**: which staff member processed each
  loan.

As always, we design this in pgModeler then apply it with Diff. This time we
will create the foreign keys using the relationship tool.

### Step 1: Open and Save As

1. Open your `stage_5_design.dbm` in pgModeler.
2. Use **File > Save As** to save it as `stage_6_design.dbm`.

### Step 2: Add the librarian table

1. Add a table named `librarian`.
2. Columns:
   - `librarian_id`, integer, primary key
   - `full_name`, varchar
   - `hire_date`, date
   - `is_manager`, boolean

### Step 3: Add the book_copy table

1. Add a table named `book_copy`.
2. Columns:
   - `copy_id`, integer, primary key
   - `acquisition_date`, date
   - `condition`, varchar

   (Do not add a book_id column by hand. The next step creates it
   automatically.)

### Step 4: Create the foreign keys with the relationship tool

This time use the **relationship tool** to draw the foreign keys and let
pgModeler create the FK columns for you. Do NOT add the FK columns manually
(adding a column by hand AND drawing a relationship would create duplicate
columns).

1. Use the relationship tool to draw a one-to-many relationship from **book**
   to **book_copy** (a book has many copies). pgModeler adds a foreign-key
   column to book_copy automatically (it will be named something like
   `book_id_book`, that is expected).
2. Use the relationship tool to draw a one-to-many relationship from
   **librarian** to **loan** (a librarian processes many loans). pgModeler
   adds a foreign-key column to loan automatically (named like
   `librarian_id_librarian`).

### Step 5: Generate the change with Diff and apply it

1. Save `stage_6_design.dbm`.
2. Open pgModeler's Diff, source = your model, target = your my_project
   database. Generate the diff. It will contain CREATE TABLE statements for
   librarian and book_copy, plus the foreign keys.
3. Review the SQL, copy it, paste it below, and run it.

In [ ]:
# Paste the SQL that pgModeler's Diff generated (CREATE TABLE librarian,
# CREATE TABLE book_copy, and the foreign keys).
stage_6_sql = r"""
--Paste the SQL that pgModeler's Diff
"""

run_ddl(stage_6_sql, project)

Note: of this error appears "Error: column "librarian_id_librarian" of relation "loan" contains null values" simply find 
ALTER TABLE public.loan ADD COLUMN librarian_id_librarian integer NOT NULL;
in the above code and delete the **NOT NULL**! save and run again!

### Step 6: Validate Stage 6, the final stage

In [ ]:
from rel_db_learning.dbm_validator import validate_exercise
from rel_db_learning.dbm_validator.exercises import PROJECT_STAGE_6_OPERATIONS

validate_exercise(
    project_db=get_project_connection(),
    spec=PROJECT_STAGE_6_OPERATIONS,
)

## Adding some staff

Let's add a few librarians so we have data to work with. Note `is_manager`
marks the managers.

In [ ]:
run_ddl("""
    INSERT INTO librarian (librarian_id, full_name, hire_date, is_manager)
    VALUES
        (1, 'Anna Vandelay', '2020-03-01', true),
        (2, 'Petros Galanis', '2022-09-15', false),
        (3, 'Lena Hoffmann', '2023-01-20', false);
""", project)
run_sql("SELECT * FROM librarian", project)

## A transaction that succeeds

Lending a book is two steps: insert a loan, and mark the book unavailable.
We wrap them in a transaction so they apply together. We lend book 5 to
member 3.

First, confirm book 5 is available:

In [ ]:
run_sql("SELECT book_id, title, is_available FROM book WHERE book_id = 5", project)

In [ ]:
run_ddl("""
    BEGIN;
        INSERT INTO loan (loan_id, book_id, member_id, loan_date, return_date, is_returned)
        VALUES (101, 5, 3, CURRENT_DATE, NULL, false);
        UPDATE book SET is_available = false WHERE book_id = 5;
    COMMIT;
""", project)

Both changes applied together. The loan exists, and the book is now
unavailable:

In [ ]:
run_sql("SELECT book_id, title, is_available FROM book WHERE book_id = 5", project)

In [ ]:
run_sql("SELECT * FROM loan WHERE loan_id = 101", project)

## A transaction that rolls back

Now the important case. Suppose we start lending book 7 but on the way through
we detect a problem and decide to cancel the whole operation. ROLLBACK undoes
everything since BEGIN as if it never happened.

First, book 7 before:

In [ ]:
run_sql("SELECT book_id, title, is_available FROM book WHERE book_id = 7", project)

In [ ]:
run_ddl("""
    BEGIN;
        INSERT INTO loan (loan_id, book_id, member_id, loan_date, return_date, is_returned)
        VALUES (102, 7, 4, CURRENT_DATE, NULL, false);
        UPDATE book SET is_available = false WHERE book_id = 7;
    ROLLBACK;
""", project)

Because we rolled back, neither change stuck. Book 7 is still available and
no loan 102 exists. The transaction was discarded entirely:

In [ ]:
run_sql("SELECT book_id, title, is_available FROM book WHERE book_id = 7", project)

In [ ]:
run_sql("SELECT * FROM loan WHERE loan_id = 102", project)

The book is available and the loan query returns no rows. This is the power
of transactions: a multi-step operation either completes fully or leaves no
trace never a half-done mess.

## Exercise

Practice a complete lending transaction. Lend book 10 to member 6: inside a
single transaction insert a loan (use loan_id 103) and set book 10 to
unavailable then COMMIT. Write the transaction below and run it.

In [ ]:
# Write a transaction that inserts loan 103 (book_id 10, member_id 6,
# loan_date CURRENT_DATE, return_date NULL, is_returned false) and sets
# book 10 unavailable, then COMMIT. Run this cell.
my_transaction = "???"
run_ddl(my_transaction, project)

In [ ]:
from rel_db_learning.widgets.self_assessment import sql_query_check

sql_query_check(
    student_sql="SELECT is_available FROM book WHERE book_id = 10",
    correct_sql="m04_nb_13_exercise_1",
    connection=project,
)

If your transaction committed correctly, book 10 is now unavailable. You can
also confirm loan 103 exists with a quick SELECT. To return to the original
data, run `reset_my_project_data()`.

## More data to experiment with

Now that your Library has all seven tables, you can load a richer dataset
that fills every table: more books and members, librarians, book copies,
reservations, returned and overdue loans, and member contact details. This
is useful for trying transactions on realistic data and it is the dataset
used in the final review notebooks.

Running this truncates the seven tables and reloads them, so it also resets
any changes you made above. Run it whenever you want a clean full Library to
play with.

In [ ]:
from rel_db_learning import reset_my_project_data_full

reset_my_project_data_full()

# A look at the new data
run_sql("SELECT COUNT(*) AS books FROM book", project)

## What's next

This was the final schema stage. Your Library now has seven tables and
models a realistic system: books and their copies, members with validated
contact details, loans processed by staff, reservations, penalties, and
views for everyday questions. In the final notebook you will review the whole
database, run a last validation and see a few showcase queries that combine
everything you have learned.